Output from octopus are usually shortened due to lack of simulation time. 
As a result, total_current and n_exc are usually cut. 
When compiling a number of files together (e.g., for color plotting), it is necessary to resample each of them. 

In [1]:
# Collect the ID of runs of interest using the database
# Fetch the corresponding files using glob
# Resample them using a defined numebr of samples. Maybe the best is to use the same as was used for laser. 
# Export the bunch into one big file compatible with GNUplot. 

In [2]:
import pandas as pd
import octopus_slabs
from octopus_slabs.Libs.libPlotting import IndexFunctional
import os
import matplotlib.pyplot as plt
import numpy as np
OctopusData=os.environ["QuantumLaPruns"]
OctopusSources=os.environ["QuantumLaPsources"]
print(OctopusData, OctopusSources)

/home/thibault.derrien/Documents/octopus.data /home/thibault.derrien/Documents/octopus-slabs


In [3]:
# Localhost
# DatabasePath=OctopusData+"/03-Bulk3D/Si-1x1x1-k-24x24x1-Mesh0.434637008855-Mesh0.434637008855-T0e0-TB09/SingleColorSummary"

# BIATRI-SRV TB09 from backups (Draco, etc)
# DatabasePath=OctopusData+"/03-Bulk3D/01-ImportingTB09"

DatabasePath=OctopusData+"/03-Bulk3D/Si-1x1x1-k-40x40x1-Angle0-Mesh0.4346370088553005-Mesh0.4346370088553005-T0e0-TB09"
%cd $DatabasePath

/home/thibault.derrien/Documents/octopus.data/03-Bulk3D/Si-1x1x1-k-40x40x1-Angle0-Mesh0.4346370088553005-Mesh0.4346370088553005-T0e0-TB09


In [4]:
data=pd.read_csv("Database.csv", "\t", header=2)
data.describe(include="all")
data=data.rename(columns={"##JobID": "JobID"})
data.dtypes


JobID                           float64
RunDate                         float64
kgrid                           float64
realmesh                        float64
TimeStep                        float64
lambda1                         float64
tau1                            float64
Efield1                         float64
PolarizationAngle               float64
CEP1                            float64
lambda2                         float64
tau2                            float64
Efield2                         float64
CEP2                            float64
delay                           float64
CrossPolar                      float64
TheoryLevel                     float64
InducedFields                   float64
ProbeFieldAmplitude_AU          float64
ProbeFieldDelay_s               float64
Functional                      float64
FrozenHxc                       float64
MoveIon                         float64
NeMax                           float64
NeFinal                         float64


In [5]:
from scipy.constants import pi
datatmp=data #copying original
datatmp=datatmp[datatmp.lambda1==2237e-9]
# datatmp=datatmp[datatmp.lambda2==745.61e-9]
datatmp=datatmp[datatmp.tau1==33.55e-15]
datatmp=datatmp[datatmp.Efield1==1.4E9]
datatmp=datatmp[datatmp.CEP1==-0.375*pi]
print(datatmp)



        JobID     RunDate  kgrid  realmesh  TimeStep   lambda1          tau1  \
11  1640785.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
12  1640649.0  20221209.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
13  1640774.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
14  1640782.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
15  1640772.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
16  1640778.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
17  1640784.0  20221212.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
18  1640775.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
19  1640790.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
20  1640780.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
21  1640777.0  20221210.0   40.0  0.434637      0.25  0.000002  3.355000e-14   
22  1640786.0  20221210.0   40.0  0.4346

In [15]:
runid=datatmp["JobID"].values

In [16]:
runid

array([1640785., 1640649., 1640774., 1640782., 1640772., 1640778.,
       1640784., 1640775., 1640790., 1640780., 1640777., 1640786.,
       1640789., 1640773., 1640791., 1640787., 1640779., 1640776.,
       1640788., 1640783., 1640781., 1645708.])

In [19]:
import glob
for JobID in np.arange(0,len(runid)):
    runid_int = int(runid[JobID])
    FileLookUp="octopus-td."+str(runid_int)+"*.log"
    ListFound=glob.glob("*/*/"+FileLookUp)
    print(FileLookUp)
    runid_folder=os.path.split(os.path.abspath(ListFound[0]))[0] #assumes only 1 finding per loop
    print(runid_folder)
    try:
        HHG=np.loadtxt(runid_folder+"/td.general/total_current", dtype='str', skiprows=4)
    except:
#           logger.error("** Error: Loading of HHG-Spectrum.csv failed.")
        logger.error("** Error: Loading of "+runid_folder+"/td.general/total_current.csv failed.")
        continue #just jump this run. Keep plotting the rest. 
    # Size of the spectrum? 
    HHG=np.asfarray(HHG)
    print("Size of HHG file: "+str(len(HHG)))
    
    # Time to resample it !
    HHG_func=interp1d(HHG[:,1], HHG[:,2], kind='nearest') #selects 


octopus-td.1640785*.log
/home/thibault.derrien/Documents/octopus.data/03-Bulk3D/Si-1x1x1-k-40x40x1-Angle0-Mesh0.4346370088553005-Mesh0.4346370088553005-T0e0-TB09/lambda2237e-9-tau33.55e-15-dt0.25-field1.4E9-CEP-0.375-theta0-SinFourth-SinPower2-2/lambda745.61e-9-tau33.55e-15-field7E7-CEP0.4-delay0e-15-SinFourth
Size of HHG file: 13001
octopus-td.1640649*.log
/home/thibault.derrien/Documents/octopus.data/03-Bulk3D/Si-1x1x1-k-40x40x1-Angle0-Mesh0.4346370088553005-Mesh0.4346370088553005-T0e0-TB09/lambda2237e-9-tau33.55e-15-dt0.25-field1.4E9-CEP-0.375-theta0-SinFourth-SinPower2-2/lambda745.61e-9-tau33.55e-15-field7E7-CEP-1.0-delay0e-15-SinFourth
Size of HHG file: 13501
octopus-td.1640774*.log
/home/thibault.derrien/Documents/octopus.data/03-Bulk3D/Si-1x1x1-k-40x40x1-Angle0-Mesh0.4346370088553005-Mesh0.4346370088553005-T0e0-TB09/lambda2237e-9-tau33.55e-15-dt0.25-field1.4E9-CEP-0.375-theta0-SinFourth-SinPower2-2/lambda745.61e-9-tau33.55e-15-field7E7-CEP-0.7-delay0e-15-SinFourth
Size of HHG fi